In [1]:
import pandas as pd
raw_file_path = '../../data/raw/shopee_reviews_dataset.jsonl'
df = pd.read_json(raw_file_path, lines=True)
print(df.head())

            id                                             review  rating  \
0  74263765409  Hương vị:thom  Chắc do bên giao hàng bị vỡ mấy...       3   
1  11104151002  Hương thơm:nhẹ nhàng Lợi ích:phục hồi cấp ẩm M...       5   
2  15888299382  Chất lượng sản phẩm:ok Đúng với mô tả:đúng  Lầ...       5   
3  81030214453  Độ tuổi sử dụng:em bes Chất lượng sản phẩm:tot...       5   
4  88484377297  Hương vị:Mix vị, Tím  Mình mua 64 gói (32 gói ...       1   

      label  
0  negative  
1  positive  
2  positive  
3  positive  
4  negative  


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

TRAIN_PATH = "data/processed/train.csv"
TEST_PATH = "data/processed/review_analysis_full_5_aspects.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# ============================================================
# 1. TRAINING DATA
# ============================================================

print("=" * 70)
print("1. TRAINING DATA - LABEL QUALITY")
print("=" * 70)

print(train["label_quality"].value_counts())
print("\nTỷ lệ:")
print(train["label_quality"].value_counts(normalize=True).round(3))


# ============================================================
# 2. MODEL PERFORMANCE
# ============================================================

print("\n" + "=" * 70)
print("2. MODEL PERFORMANCE")
print("=" * 70)

labels = ["positive", "negative", "neutral", "none"]

y_true = test["label_quality"]
y_pred = test["pred_quality"]

print(confusion_matrix(y_true, y_pred, labels=labels))

print(
    classification_report(
        y_true,
        y_pred,
        labels=labels,
        digits=3
    )
)


# ============================================================
# 3. NONE -> NEGATIVE
# ============================================================

print("\n" + "=" * 70)
print("3. QUALITY NONE -> NEGATIVE")
print("=" * 70)

none_total = (y_true == "none").sum()
none_to_negative = (
    (y_true == "none") &
    (y_pred == "negative")
).sum()

print("Quality = none:", none_total)
print("Bị dự đoán thành negative:", none_to_negative)

if none_total > 0:
    print(
        "Tỷ lệ:",
        round(none_to_negative / none_total * 100, 2),
        "%"
    )


# ============================================================
# 4. TÌM CÁC REVIEW KHẢ NGHI
# ============================================================

print("\n" + "=" * 70)
print("4. REVIEW KHẢ NGHI: QUALITY NONE -> NEGATIVE")
print("=" * 70)

suspect = test[
    (test["label_quality"] == "none") &
    (test["pred_quality"] == "negative")
].copy()

# Các từ khóa đại diện cho aspect khác
keywords = (
    "đóng gói|đóng hàng|"
    "giao hàng|vận chuyển|ship|shipper|"
    "chăm sóc khách hàng|CSKH|tư vấn|"
    "quà tặng|khuyến mãi"
)

suspect_other_aspect = suspect[
    suspect["review_text"]
    .str.contains(keywords, case=False, na=False)
]

print(
    "Tổng lỗi None -> Negative:",
    len(suspect)
)

print(
    "Trong đó có dấu hiệu aspect khác:",
    len(suspect_other_aspect)
)

if len(suspect) > 0:
    print(
        "Tỷ lệ lỗi có dấu hiệu aspect khác:",
        round(
            len(suspect_other_aspect) /
            len(suspect) * 100,
            2
        ),
        "%"
    )


# ============================================================
# 5. KẾT LUẬN TỰ ĐỘNG
# ============================================================

print("\n" + "=" * 70)
print("5. KẾT LUẬN")
print("=" * 70)

none_rate = (
    none_to_negative / none_total
    if none_total > 0 else 0
)

other_aspect_rate = (
    len(suspect_other_aspect) / len(suspect)
    if len(suspect) > 0 else 0
)

macro_f1 = classification_report(
    y_true,
    y_pred,
    labels=labels,
    output_dict=True
)["macro avg"]["f1-score"]

print(f"Macro F1: {macro_f1:.3f}")
print(f"None -> Negative: {none_rate:.1%}")
print(f"Lỗi có dấu hiệu aspect khác: {other_aspect_rate:.1%}")


if macro_f1 >= 0.70 and other_aspect_rate < 0.50:
    print("\n🟢 KẾT LUẬN:")
    print("Model đủ tốt cho đồ án.")
    print("Nên CHẤP NHẬN giới hạn hiện tại và không train lại.")

elif macro_f1 >= 0.70 and other_aspect_rate >= 0.50:
    print("\n🟡 KẾT LUẬN:")
    print("Model sentiment khá tốt nhưng aspect separation còn hạn chế.")
    print("Nên cải thiện DATA/ANNOTATION và train lại nếu còn thời gian.")

else:
    print("\n🔴 KẾT LUẬN:")
    print("Model chưa đủ tốt.")
    print("Nên ưu tiên cải thiện dataset và train lại.")

                                                                                                                                                                    review_text label_quality pred_quality
                Bao bì:ok Làm đẹp:ok Hương thơm:ok  mình chưa dùng sản phẩm nhưng bóc ra là thấy k vui rồi , đóng hàng kiểu gì mà mở ra lung tung hết , góp ý với shop vậy thôi          none     negative
                                                                                    Đóng hàng quá tệ. Móp méo vỡ nát tùm lum. Làm ăn cẩu thả. Đuổi việc caia người đóng hàng đi          none     negative
                                                      Trả hàng còn nguyên mà dám bảo khách dùng ra bóc rồi bảo cung cấp bằng chứng trước khi vận chuyển thì không cung cấp được          none     negative
                                                                                                                               Chẳng bao giờ thấy hàng khuyến mãi như quảng cáo          non

In [6]:
text_column = 'review'
df_clean = df.dropna(subset=[text_column]).copy()
df_clean['word_count'] = df_clean[text_column].apply(lambda x: len(str(x).split()))
df_clean = df_clean[(df_clean['word_count'] >= 5) & (df_clean['word_count'] <= 40)]
print(len(df_clean))

7697


In [7]:
SAMPLE_SIZE = 3000
if len(df_clean) > SAMPLE_SIZE:
    # random_state=42 giúp đảm bảo mỗi lần chạy lại code sẽ ra đúng 3000 câu giống hệt nhau
    df_sampled = df_clean.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    df_sampled = df_clean.reset_index(drop=True)

# Chỉ giữ lại đúng cột nội dung để tiết kiệm bộ nhớ, đổi tên thành 'review_text' cho chuẩn form
df_final = df_sampled[[text_column]].rename(columns={text_column: 'review_text'})

# 4. Xuất file chuẩn bị cho bước Auto-Labeling
output_path = '../../data/processed/shopee_sampled_for_labeling.csv'
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ Đã xuất {len(df_final)} dòng ra file: {output_path}")
display(df_final.head())

✅ Đã xuất 3000 dòng ra file: ../../data/processed/shopee_sampled_for_labeling.csv


,review_text
0,"Shop giao hàng nhanh, đúng số lượng và mô tả, ..."
1,"Hơi thất vọng vì ko còn là mẫu cũ, dạng sữa, g..."
2,Bao bì/Mẫu mã:đúng Hương vị:mix vị Hàng giao ...
3,"Hình ảnh mang tính chất nhận xu, sản phẩm y hì..."
4,Lợi ích:sạch da Làm đẹp:ổn Kinh nghiệm sử dụng...
